In [10]:
import pandas as pd
import datasets
import numpy as np
from sentence_transformers import SentenceTransformer
import os
import json
import re

# --- Config ---
embedder_name = "Qwen3-Embedding-0.6B"
dataset_name = "dapo-math-17k"
num_samples = 100  # Change to 17000 for full dataset

# --- Helper Functions ---
def gaussian_kernel(x, y, sigma=1.0):
    dist = np.linalg.norm(x - y)
    return np.exp(-dist ** 2 / (2 * sigma ** 2))

def extract_question_from_prompt(prompt_content, template):
    split_token = '\n\n'
    if split_token in template:
        parts = template.split(split_token)
        if '{{question}}' in parts[-2]:
            prefix = split_token.join(parts[:-2]) + split_token
            question = prompt_content.replace(prefix, '').strip()
            if '\n\n' in question:
                question = question.split('\n\n')[0].strip()
            return question
    matches = re.split(r'\n\n', prompt_content)
    if len(matches) > 1:
        return matches[-2].strip()
    return prompt_content.strip()

# --- Load Data ---
dapo_file_path = '/home/hieunt/verl/data/dapo-math-17k.parquet'
parquet_df = datasets.load_dataset('parquet', data_files=[dapo_file_path])['train']

# --- Prepare Prompts and Indices ---
prompt_contents = [prompt[0]['content'] for prompt in parquet_df['prompt'][:num_samples]]
prompt_indices = [extra['index'] for extra in parquet_df['extra_info'][:num_samples]]
prompt_content_template = 'Solve the following math problem step by step. The last line of your response should be of the form Answer: $Answer (without quotes) where $Answer is the answer to the problem.\n\n{{question}}\n\nRemember to put your answer on its own line after "Answer:".'

# --- Extract Questions ---
questions = [extract_question_from_prompt(pc, prompt_content_template) for pc in prompt_contents]

# --- Compute Embeddings ---
model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B")
embeddings = model.encode(questions, prompt_name="query")

# --- Compute Pairwise Gaussian Kernel Distances ---
pairwise_matrix = np.zeros((num_samples, num_samples), dtype=np.float32)
for i in range(num_samples):
    for j in range(num_samples):
        sim = gaussian_kernel(embeddings[i], embeddings[j], sigma=1.0)
        pairwise_matrix[i, j] = sim

# --- Save Outputs with Naming Convention ---
os.makedirs('data/embeddings', exist_ok=True)
matrix_path = f'data/embeddings/pairwise_{embedder_name}_{dataset_name}_{num_samples}_matrix.npy'
indices_path = f'data/embeddings/pairwise_{embedder_name}_{dataset_name}_{num_samples}_indices.json'
np.save(matrix_path, pairwise_matrix)
with open(indices_path, 'w') as f:
    json.dump(prompt_indices, f, indent=2)

print(f"Saved pairwise matrix to {matrix_path} and indices to {indices_path}")

Saved pairwise matrix to data/embeddings/pairwise_Qwen3-Embedding-0.6B_dapo-math-17k_100_matrix.npy and indices to data/embeddings/pairwise_Qwen3-Embedding-0.6B_dapo-math-17k_100_indices.json


In [11]:
def load_pairwise_matrix(matrix_path, indices_path):
    """
    Loads the pairwise distance matrix and prompt indices, and returns a dictionary
    mapping each prompt index to its corresponding row (as a numpy array) in the matrix.

    Args:
        matrix_path (str): Path to the .npy file containing the pairwise matrix.
        indices_path (str): Path to the .json file containing the prompt indices.

    Returns:
        dict: {prompt_index: np.ndarray}
    """
    matrix = np.load(matrix_path)
    with open(indices_path, 'r') as f:
        indices = json.load(f)
    return {idx: matrix[i] for i, idx in enumerate(indices)}

In [ ]:
load

Saved pairwise matrix to data/embeddings/pairwise_distances_test.npy and indices to data/embeddings/pairwise_indices_test.json


'Solve the following math problem step by step. The last line of your response should be of the form Answer: $Answer (without quotes) where $Answer is the answer to the problem.\n\nFind the sum of all possible values of $a + b$ where $a$ and $b$ are nonnegative integers such that $4^a + 2^b + 5$ is a perfect square.\n\nRemember to put your answer on its own line after "Answer:".'